In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [2]:
import torch
from torch.optim import Adam
from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn

/u/bing/sbi_bmode/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def build_inference(prior, density_estimator_type, hidden_features,
                     num_transforms, num_blocks, embedding_net=None):
    '''
    Construct a fresh SNPE object with a given architecture. Use the exact
    same args here as when you originally trained the model whose
    state_dict you're loading -- shape mismatches will raise immediately
    in load_state_dict, which is a good, loud failure mode.
    '''
    if embedding_net is None:
        embedding_net = torch.nn.Identity()

    net_builder = posterior_nn(
        model=density_estimator_type,
        hidden_features=hidden_features,
        num_transforms=num_transforms,
        num_block=num_blocks,          # NOTE: singular, matches your current scripts.
                                        # If this raises TypeError on your sbi version,
                                        # try num_blocks (plural) instead -- that's the
                                        # kwarg name in sbi 0.27.x.
        dropout_probability=0.0,
        use_batch_norm=False,
        embedding_net=embedding_net,
    )
    return SNPE(prior, density_estimator=net_builder)

def save_net_state_dict(inference, path):
    '''
    Call after inference.train() has completed. Saves only the density
    estimator's weights, not the whole inference object (no cached sims,
    no optimizer state, no prior).
    '''
    torch.save(inference._neural_net.state_dict(), path)
    print(f'Saved state_dict to {path}')
    
def warm_start_from_state_dict(inference, theta, x, state_dict_path, learning_rate,
                                proposal=None):
    '''
    Warm-start `inference`'s density estimator from a saved state_dict, then
    stage it to resume training at `learning_rate`.

    Must be called:
      - AFTER inference.append_simulations(theta, x, proposal=proposal)
      - BEFORE the real inference.train(..., resume_training=True) call

    `theta`/`x` are only used to determine the net's input/output shapes via
    the throwaway epoch below -- they don't need to be the full dataset.
    '''
    # Throwaway epoch: this is the part that's easy to skip and then hit
    # "ValueError: This neural network has already been trained" later.
    # .train() is what actually builds _neural_net, train_indices/val_indices,
    # and the optimizer on its first call -- calling _build_neural_net()
    # directly and assigning inference._neural_net skips all of that.
    inference.train(
        training_batch_size=200,
        learning_rate=learning_rate,
        max_num_epochs=1,          # sbi requires a positive int here
        stop_after_epochs=10**6,   # effectively disables early stopping for this dummy epoch
        show_train_summary=False,
    )

    state_dict = torch.load(state_dict_path, map_location='cpu')
    inference._neural_net.load_state_dict(state_dict)

    # Reset bookkeeping so the *next* real .train() call starts a fresh
    # optimizer / epoch count / early-stopping window at the new learning
    # rate, instead of resuming the throwaway epoch's state.
    inference.optimizer = Adam(inference._neural_net.parameters(), lr=learning_rate)
    inference.epoch = 0
    inference._val_loss = float('inf')

    return inference

def check_warm_start_sane(inference, loaded_state_dict, fresh_inference):
    '''
    Two sanity checks to run right after warm_start_from_state_dict, before
    trusting a real (expensive) training run:
      1. Loaded weights are byte-identical to what was saved.
      2. Loaded weights differ from a same-shape freshly-initialized net
         (i.e. loading actually overwrote the throwaway epoch, not a no-op).

    `fresh_inference` should be a second inference object built the same way,
    appended with the same simulations, and run through the same throwaway
    .train() call as warm_start_from_state_dict does -- but WITHOUT loading
    the state_dict.
    '''
    mismatch = [k for k, v in inference._neural_net.state_dict().items()
                if not torch.equal(v, loaded_state_dict[k])]
    assert not mismatch, f'state_dict did not load correctly: {mismatch}'
    print('[check] warm-started weights match saved state_dict exactly: OK')

    same_as_fresh = all(
        torch.equal(a, b) for a, b in zip(
            inference._neural_net.state_dict().values(),
            fresh_inference._neural_net.state_dict().values()))
    assert not same_as_fresh, 'warm-started net looks identical to a fresh random init!'
    print('[check] warm-started weights differ from a fresh random init: OK')

In [4]:
from torch.distributions import MultivariateNormal

In [5]:
A_TRUE = torch.tensor([[1.0, 0.5], [0.2, 1.0]])
def simulate(theta, noise_std):
    return theta @ A_TRUE.T + noise_std * torch.randn(theta.shape[0], 2)

prior = MultivariateNormal(torch.zeros(2), torch.eye(2))
ARCH = dict(density_estimator_type='maf', hidden_features=20,
            num_transforms=3, num_blocks=2)

In [6]:
# 1. "low-fidelity" pretrain + save
theta_lo, x_lo = prior.sample((1000,)), None
theta_lo = prior.sample((1000,))
x_lo = simulate(theta_lo, noise_std=0.5)

inf_lo = build_inference(prior, **ARCH)
inf_lo.append_simulations(theta_lo, x_lo)
inf_lo.train(training_batch_size=200, learning_rate=5e-4,
             stop_after_epochs=10, max_num_epochs=50, show_train_summary=False)
save_net_state_dict(inf_lo, '/tmp/net_state_dict.pt')

/tmp/ipykernel_3435217/2685789299.py:12: UserWarning: Unknown kwargs passed to ConditionalFlowConfig: {'num_block'}. These will be forwarded to the underlying builder. If this is unintentional, check for typos.
  net_builder = posterior_nn(


 Training neural network. Epochs trained: 51Saved state_dict to /tmp/net_state_dict.pt


In [7]:
# 2. warm start on "high-fidelity" data
theta_hi = prior.sample((1000,))
x_hi = simulate(theta_hi, noise_std=0.05)

inf_warm = build_inference(prior, **ARCH)
inf_warm.append_simulations(theta_hi, x_hi)
inf_warm = warm_start_from_state_dict(inf_warm, theta_hi, x_hi,
                                       '/tmp/net_state_dict.pt', learning_rate=1e-4)

 Training neural network. Epochs trained: 2

/tmp/ipykernel_3435217/2685789299.py:12: UserWarning: Unknown kwargs passed to ConditionalFlowConfig: {'num_block'}. These will be forwarded to the underlying builder. If this is unintentional, check for typos.
  net_builder = posterior_nn(


In [8]:
# 3. sanity checks
loaded_state = torch.load('/tmp/net_state_dict.pt', map_location='cpu')
inf_fresh = build_inference(prior, **ARCH)
inf_fresh.append_simulations(theta_hi, x_hi)
inf_fresh.train(training_batch_size=200, learning_rate=5e-4,
                 max_num_epochs=1, stop_after_epochs=10**6, show_train_summary=False)
check_warm_start_sane(inf_warm, loaded_state, inf_fresh)

 Training neural network. Epochs trained: 2[check] warm-started weights match saved state_dict exactly: OK
[check] warm-started weights differ from a fresh random init: OK


/tmp/ipykernel_3435217/2685789299.py:12: UserWarning: Unknown kwargs passed to ConditionalFlowConfig: {'num_block'}. These will be forwarded to the underlying builder. If this is unintentional, check for typos.
  net_builder = posterior_nn(


In [9]:
# 4. resume real training
inf_warm.train(training_batch_size=200, learning_rate=1e-4,
                stop_after_epochs=10, max_num_epochs=50,
                show_train_summary=False, resume_training=True)
print('final val loss:', inf_warm.summary['validation_loss'][-1])

 Training neural network. Epochs trained: 51final val loss: 1.4347779846191406
